# Anomalous protein–protein interactions in multivalent salt solution

*Coralie Pasquier, Mario Vazdar, Jan Forsman, Pavel Jungwirth, and Mikael Lund,
J. Phys. Chem. B 121, 3000–3006 (2017), [doi:10.1021/acs.jpcb.7b01051](https://doi.org/10.1021/acs.jpcb.7b01051)*

Two human serum albumin (HSA) molecules move along the axis of a periodic cylinder with explicit
trivalent yttrium ions, which move freely. Sites interact through a combined Debye–Hückel/Lennard-Jones
potential, and the protein–protein distance distribution gives the potential of mean force (PMF).

<img src="../figures/snapshot.png" style="width: 800px;"/>

## 01 – Monte Carlo simulations

**Optional.** This notebook compiles Faunus and runs the Metropolis Monte Carlo simulations; this takes
days. Its result, `data/processed/pmf.csv`, is already in the repository, so to reproduce the figures
run only `02-pmf.ipynb` (or `uv run analysis`).

The program `data/raw/faunus/twobody.cpp` is written against the 2017 Faunus API (commit `bdf6c7f7`)
and needs a C++ compiler and CMake. Everything is built and run in `build/`, which git ignores.


## Conditions

- **Implicit or explicit Na⁺ and Cl⁻.** Y³⁺ is always explicit. *Implicit*: only Y³⁺ is in the box, and
  the Debye length accounts for the Cl⁻ from YCl₃, the Na⁺ counterions, and any added 1:1 salt.
  *Explicit*: Y³⁺, Cl⁻ and Na⁺ are all in the box, and added salt is ignored.
- **Specific or non-specific.** *Specific* adds a short-range Y³⁺–carboxylate (Asp, Glu, C-terminus)
  attraction. The published PMFs use implicit salt with specific interactions.
- Set `run_equilibration` and `run_production` to `True` to run.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np

from analysis.faunus_input import faunus_input
from analysis.pmf import pmf_from_rdf, write_pmf

ROOT = Path.cwd().parent
BUILD = ROOT / "build"
RUNS = BUILD / "runs"
STRUCTURE = ROOT / "data" / "raw" / "faunus" / "hsa-ph7.aam"

added_salt = [0.0, 0.1, 0.2]  # 1:1 salt, mol/l
n_yttrium = [1, 8, 16, 160]
explicit_ions = False
specific = True

run_equilibration = False
run_production = False
micro_eq, micro_prod = 100_000, 10_000_000

## Compile Faunus

Skip unless you want to run new simulations.

In [ ]:
%%bash -s "$BUILD" "$ROOT"
set -e
mkdir -p "$1/playground" && cd "$1"
cp "$2/data/raw/faunus/twobody.cpp" playground/
echo 'fau_example(twobody "./" twobody.cpp)' > playground/CMakeLists.txt
if [ ! -d faunus ]; then
    git clone https://github.com/mlund/faunus.git
    git -C faunus checkout bdf6c7f7
fi
cd faunus
CXX=c++ CC=cc cmake . -DCMAKE_BUILD_TYPE=Release -DENABLE_APPROXMATH=on -DMYPLAYGROUND="$1/playground" > /dev/null
make twobody -j4

## Run

One directory per condition under `build/runs/`. Production runs go to Slurm (`sbatch`) when it is
available, otherwise they run locally.

Output per run: `eq`/`out` (standard output), `rdf.dat` (protein–protein distance distribution),
`confout.pqr` (final configuration, e.g. for VMD).

In [ ]:
def run_name(n, salt):
    ions = "explicit" if explicit_ions else "implicit"
    lj = "specific" if specific else "nonspecific"
    return f"Ns{n}-{ions}-{lj}-Cs{salt}"


def write_input(rundir, n, salt, micro):
    structure = Path("..", "..", "..", STRUCTURE.relative_to(ROOT))
    d = faunus_input(
        n,
        salt,
        explicit_ions=explicit_ions,
        specific=specific,
        micro=micro,
        structure=structure.as_posix(),
    )
    (rundir / "hsa.json").write_text(json.dumps(d, indent=4), encoding="utf-8")


program = BUILD / "playground" / "twobody"
for salt in added_salt:
    for n in n_yttrium:
        rundir = RUNS / run_name(n, salt)
        rundir.mkdir(parents=True, exist_ok=True)
        if run_equilibration:
            (rundir / "state").unlink(missing_ok=True)
            write_input(rundir, n, salt, micro_eq)
            with (rundir / "eq").open("w", encoding="utf-8") as log:
                subprocess.run([program], cwd=rundir, stdout=log, check=True)
        if run_production:
            write_input(rundir, n, salt, micro_prod)
            if shutil.which("sbatch"):
                job = f"#!/bin/bash\n#SBATCH -N 1\n#SBATCH -n 1\n#SBATCH -t 168:00:00\n{program} > out\n"
                (rundir / "submit.sh").write_text(job, encoding="utf-8")
                subprocess.run(["sbatch", "submit.sh"], cwd=rundir, check=True)
            else:
                with (rundir / "out").open("w", encoding="utf-8") as log:
                    subprocess.run([program], cwd=rundir, stdout=log, check=True)

## Collect PMFs

Converts each `rdf.dat` to w(r) and writes `build/runs/pmf.csv`. Compare it with, then copy it over,
`data/processed/pmf.csv` to use it in `02-pmf.ipynb`.

In [ ]:
curves = {}
for salt in added_salt:
    for n in n_yttrium:
        rdf = RUNS / run_name(n, salt) / "rdf.dat"
        if rdf.is_file():
            r, g = np.loadtxt(rdf, usecols=(0, 1), unpack=True)
            curves[(salt, n)] = (r, pmf_from_rdf(r, g))
if curves:
    write_pmf(RUNS / "pmf.csv", curves)
print(f"{len(curves)} PMFs collected")